Dental clinic conversation (audio) - appgpt

In [13]:
# =====================================================
# MEDICAL SPEECH AI PROJECT
# WHISPER FINE-TUNING PIPELINE
# FINAL STABLE SUBMISSION VERSION
# TRANSFORMERS 4.41.x
# =====================================================

# =====================================================
# REQUIRED INSTALLS
# =====================================================

# pip install transformers==4.41.2
# pip install datasets==2.19.1
# pip install accelerate==0.30.1
# pip install evaluate==0.4.2
# pip install jiwer==3.0.4
# pip install sentencepiece
# pip install librosa
# pip install soundfile
# pip install pandas
# pip install torch

# =====================================================
# IMPORTS
# =====================================================

from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch
import evaluate

from datasets import Dataset
from datasets import DatasetDict

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

# =====================================================
# CONFIG
# =====================================================

DEBUG_MODE = True

DEBUG_SAMPLES = 3

TARGET_SAMPLE_RATE = 16000

SEED = 42

np.random.seed(SEED)

# =====================================================
# PROJECT ROOT
# =====================================================

BASE_DIR = Path().resolve()

DATA_DIR = BASE_DIR / "Medical Speech, Transcription, and Intent"

CSV_PATH = DATA_DIR / "overview-of-recordings.csv"

RECORDINGS_DIR = DATA_DIR / "recordings"

TRAIN_DIR = RECORDINGS_DIR / "train"

TEST_DIR = RECORDINGS_DIR / "test"

VALIDATE_DIR = RECORDINGS_DIR / "validate"

MODEL_DIR = BASE_DIR / "models"

MODEL_DIR.mkdir(exist_ok=True)

# =====================================================
# SECTION 1 — DATASET DETAILS
# =====================================================

print("=" * 60)
print("SECTION 1 — DATASET DETAILS")
print("=" * 60)

print("\nLOADING CSV...")

df = pd.read_csv(CSV_PATH)

print("\nTOTAL CSV ROWS:", len(df))

print("\nCSV COLUMNS:")

for c in df.columns:
    print("-", c)

# =====================================================
# BUILD AUDIO INDEX
# =====================================================

print("\nINDEXING AUDIO FILES...")

file_map = {}

split_lookup = {}

folders = [
    ("train", TRAIN_DIR),
    ("test", TEST_DIR),
    ("validate", VALIDATE_DIR)
]

for split_name, folder in folders:

    wavs = list(folder.glob("*.wav"))

    print(f"{split_name}: {len(wavs)} wav files")

    for wav_file in wavs:

        file_map[wav_file.name] = str(
            wav_file.resolve()
        )

        split_lookup[wav_file.name] = split_name

print("\nTOTAL AUDIO FILES:", len(file_map))

# =====================================================
# CSV ↔ WAV MATCHING
# =====================================================

print("\nVERIFYING CSV ↔ WAV MATCHING...")

matched_rows = []

missing_files = []

for _, row in df.iterrows():

    file_name = str(
        row["file_name"]
    ).strip()

    if file_name in file_map:

        matched_rows.append({

            "audio": file_map[file_name],

            "text": str(
                row["phrase"]
            ).strip(),

            "split": split_lookup[file_name]
        })

    else:

        missing_files.append(file_name)

print("\nMATCHED FILES:", len(matched_rows))

print("MISSING FILES:", len(missing_files))

if len(missing_files) > 0:

    print("\nFIRST 10 MISSING FILES:")

    for m in missing_files[:10]:
        print(m)

# =====================================================
# DATASET DISTRIBUTION
# =====================================================

distribution = Counter(
    [x["split"] for x in matched_rows]
)

print("\nDATASET DISTRIBUTION:")

for k, v in distribution.items():
    print(f"{k}: {v}")

# =====================================================
# SECTION 2 — PREPROCESSING
# =====================================================

print("\n" + "=" * 60)
print("SECTION 2 — PREPROCESSING")
print("=" * 60)

# =====================================================
# SPLIT DATASET
# IMPORTANT:
# USE ORIGINAL DATASET SPLITS
# DO NOT RANDOM SPLIT
# =====================================================

train_rows = [
    x for x in matched_rows
    if x["split"] == "train"
]

validation_rows = [
    x for x in matched_rows
    if x["split"] == "validate"
]

test_rows = [
    x for x in matched_rows
    if x["split"] == "test"
]

print("\nTRAIN ROWS:", len(train_rows))
print("VALIDATION ROWS:", len(validation_rows))
print("TEST ROWS:", len(test_rows))

# =====================================================
# DEBUG MODE
# =====================================================

if DEBUG_MODE:

    print("\nDEBUG MODE ENABLED")

    train_rows = train_rows[:DEBUG_SAMPLES]

    validation_rows = validation_rows[:DEBUG_SAMPLES]

    test_rows = test_rows[:DEBUG_SAMPLES]

# =====================================================
# HUGGINGFACE DATASET
# =====================================================

dataset = DatasetDict({

    "train": Dataset.from_list(
        train_rows
    ),

    "validation": Dataset.from_list(
        validation_rows
    )
})

# =====================================================
# LOAD WHISPER
# =====================================================

print("\nLOADING WHISPER PROCESSOR...")

processor = WhisperProcessor.from_pretrained(
    "openai/whisper-base"
)

print("\nLOADING WHISPER MODEL...")

model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-base"
)

model.generation_config.language = "english"

model.generation_config.task = "transcribe"

# =====================================================
# AUDIO LOADER
# =====================================================

def load_audio(audio_path):

    audio, sr = sf.read(audio_path)

    audio = audio.astype(np.float32)

    # =============================================
    # STEREO → MONO
    # =============================================

    if audio.ndim > 1:

        audio = np.mean(
            audio,
            axis=1
        )

    # =============================================
    # NORMALIZE
    # =============================================

    audio = audio / (
        np.max(np.abs(audio)) + 1e-8
    )

    return audio, sr

# =====================================================
# RESAMPLE
# =====================================================

def resample_audio(

    audio,

    sr,

    target_sr=TARGET_SAMPLE_RATE
):

    if sr != target_sr:

        audio = librosa.resample(

            audio,

            orig_sr=sr,

            target_sr=target_sr
        )

        sr = target_sr

    return audio, sr

# =====================================================
# PREPROCESS FUNCTION
# =====================================================

def preprocess(batch):

    # =============================================
    # LOAD AUDIO
    # =============================================

    audio, sr = load_audio(
        batch["audio"]
    )

    # =============================================
    # RESAMPLE
    # =============================================

    audio, sr = resample_audio(
        audio,
        sr
    )

    # =============================================
    # WHISPER FEATURES
    # =============================================

    input_features = processor.feature_extractor(

        audio,

        sampling_rate=TARGET_SAMPLE_RATE,

        return_tensors="pt"

    ).input_features[0]

    # =============================================
    # TOKENIZE TEXT
    # =============================================

    labels = processor.tokenizer(

        batch["text"]

    ).input_ids

    # =============================================
    # SAVE
    # =============================================

    batch["input_features"] = input_features

    batch["labels"] = labels

    return batch

# =====================================================
# PREPROCESS DATASET
# =====================================================

print("\nPREPROCESSING DATASET...")

dataset = dataset.map(

    preprocess,

    remove_columns=dataset["train"].column_names
)

# =====================================================
# SECTION 3 — EVALUATION METRIC
# =====================================================

print("\n" + "=" * 60)
print("SECTION 3 — WER EVALUATION")
print("=" * 60)

wer_metric = evaluate.load("wer")

# =====================================================
# COMPUTE METRICS
# =====================================================

def compute_metrics(pred):

    pred_ids = pred.predictions

    label_ids = pred.label_ids

    # =============================================
    # REPLACE -100
    # =============================================

    label_ids[label_ids == -100] = (
        processor.tokenizer.pad_token_id
    )

    # =============================================
    # DECODE
    # =============================================

    pred_str = processor.tokenizer.batch_decode(

        pred_ids,

        skip_special_tokens=True
    )

    label_str = processor.tokenizer.batch_decode(

        label_ids,

        skip_special_tokens=True
    )

    # =============================================
    # WER
    # =============================================

    wer = wer_metric.compute(

        predictions=pred_str,

        references=label_str
    )

    return {
        "wer": wer
    }

# =====================================================
# DATA COLLATOR
# =====================================================

class WhisperDataCollator:

    def __init__(self, processor):

        self.processor = processor

    def __call__(self, features):

        # =========================================
        # INPUT FEATURES
        # =========================================

        input_features = [

            {
                "input_features": f["input_features"]
            }

            for f in features
        ]

        batch = self.processor.feature_extractor.pad(

            input_features,

            return_tensors="pt"
        )

        # =========================================
        # LABEL FEATURES
        # =========================================

        label_features = [

            {
                "input_ids": f["labels"]
            }

            for f in features
        ]

        labels_batch = self.processor.tokenizer.pad(

            label_features,

            return_tensors="pt"
        )

        labels = labels_batch["input_ids"]

        # =========================================
        # PAD TOKEN → -100
        # =========================================

        labels[
            labels == self.processor.tokenizer.pad_token_id
        ] = -100

        batch["labels"] = labels

        return batch

# =====================================================
# COLLATOR INSTANCE
# =====================================================

data_collator = WhisperDataCollator(
    processor
)

# =====================================================
# SECTION 4 — TRAINING
# =====================================================

print("\n" + "=" * 60)
print("SECTION 4 — WHISPER FINE-TUNING")
print("=" * 60)

training_args = Seq2SeqTrainingArguments(

    output_dir=str(MODEL_DIR),

    per_device_train_batch_size=1,

    per_device_eval_batch_size=1,

    learning_rate=1e-5,

    num_train_epochs=1,

    eval_strategy="epoch",

    save_strategy="epoch",

    logging_steps=1,

    predict_with_generate=True,

    fp16=torch.cuda.is_available(),

    report_to="none"
)

# =====================================================
# TRAINER
# =====================================================

trainer = Seq2SeqTrainer(

    model=model,

    args=training_args,

    train_dataset=dataset["train"],

    eval_dataset=dataset["validation"],

    data_collator=data_collator,

    processing_class=processor.feature_extractor,

    compute_metrics=compute_metrics
)

# =====================================================
# TRAIN
# =====================================================

print("\nSTARTING TRAINING...")

trainer.train()

# =====================================================
# EVALUATION
# =====================================================

print("\nRUNNING EVALUATION...")

metrics = trainer.evaluate()

print("\nFINAL METRICS:")

print(metrics)

# =====================================================
# SAVE MODEL
# =====================================================

print("\nSAVING MODEL...")

trainer.save_model(str(MODEL_DIR))

processor.save_pretrained(str(MODEL_DIR))

# =====================================================
# SECTION 5 — INFERENCE PIPELINE
# =====================================================

print("\n" + "=" * 60)
print("SECTION 5 — INFERENCE")
print("=" * 60)

sample = test_rows[0]

audio, sr = load_audio(
    sample["audio"]
)

audio, sr = resample_audio(
    audio,
    sr
)

inputs = processor(

    audio,

    sampling_rate=TARGET_SAMPLE_RATE,

    return_tensors="pt"
).input_features

# =====================================================
# GENERATE
# =====================================================

predicted_ids = model.generate(
    inputs
)

transcript = processor.batch_decode(

    predicted_ids,

    skip_special_tokens=True

)[0]

# =====================================================
# OUTPUT
# =====================================================

print("\nTRANSCRIPT:")

print(transcript)

# =====================================================
# NER HANDOFF
# =====================================================

ner_payload = {
    "text": transcript
}

print("\nNER PAYLOAD:")

print(ner_payload)

# =====================================================
# DONE
# =====================================================

print("\nPIPELINE COMPLETE")

SECTION 1 — DATASET DETAILS

LOADING CSV...

TOTAL CSV ROWS: 6661

CSV COLUMNS:
- audio_clipping
- audio_clipping:confidence
- background_noise_audible
- background_noise_audible:confidence
- overall_quality_of_the_audio
- quiet_speaker
- quiet_speaker:confidence
- speaker_id
- file_download
- file_name
- phrase
- prompt
- writer_id

INDEXING AUDIO FILES...
train: 381 wav files
test: 5895 wav files
validate: 385 wav files

TOTAL AUDIO FILES: 6661

VERIFYING CSV ↔ WAV MATCHING...

MATCHED FILES: 6661
MISSING FILES: 0

DATASET DISTRIBUTION:
test: 5895
train: 381
validate: 385

SECTION 2 — PREPROCESSING

TRAIN ROWS: 381
VALIDATION ROWS: 385
TEST ROWS: 5895

DEBUG MODE ENABLED

LOADING WHISPER PROCESSOR...

LOADING WHISPER MODEL...


Loading weights: 100%|██████████| 245/245 [00:00<00:00, 6620.74it/s]



PREPROCESSING DATASET...


Map: 100%|██████████| 3/3 [00:00<00:00, 88.25 examples/s]



SECTION 3 — WER EVALUATION

SECTION 4 — WHISPER FINE-TUNING

STARTING TRAINING...


...

d:\Math_Final-Project\venv\lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Wer
7.956104,7.416801,1,0.633333



FINAL METRICS:
{'eval_loss': 7.416801452636719, 'eval_wer': 0.6333333333333333}

SAVING MODEL...


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.05s/it]



SECTION 5 — INFERENCE

TRANSCRIPT:
 when I remember her eye field.

NER PAYLOAD:
{'text': ' when I remember her eye field.'}

PIPELINE COMPLETE
